# Modeling
## March 2, 2026

# Data Dictionary

- id - unique ID for workout
- user_id - unique ID for athlete who did the workout
- sport_type - sport for the workout (~40 unique)
- sport_type_grouped - groups workouts into main groups
- speed_mph - miles / elapsed time in hours
- distance - distance in meters
- miles - distance in miles
- kilometers - distance in kilometers
- moving_time - seconds of active moving time (pauses for red light, water break, etc)
- elapsed_time - total seconds for entire workout
- moving_minutes - minutes of active moving time
- elapsed_minutes - minutes for entire workout
- moving_time_per - moving_minutes / elapsed_minutes
- total_elevation_gain - meters of climbing
- meters_per_km - avg meters of climbing per kilometer
- feet_per_mile - avg feet of climbing per mile (for the Americans lol)
- commute - boolean flag is user marked the activity as a commute (like when Oliver bikes to class)
- manual - flag for if the workout was generated by a tracking device or if user manually entered the details
- has_gear - boolean for if user indicated which shoes/bike they used for the workout
- suffer_score - Strava metric used to describe how tough the workout is; function of heart rate and total time
- kudos_count - how many "likes" the workout received on Strava
- device_name - name used to record the workout
- start_date - date of workout
- hour - hour of workout (start)
- day_part - morning vs afternoon vs evening vs night (start)
- month - month of workout
- dayofweek - day of week of workout
- is_weekend - boolean for if dayofweek == Saturday or Sunday
- is_northern_hemisphere - start_lat > 0
- num_turns - number of turns in the GPS trace
- turns_per_mile - num_turns / miles
- wobble - how wiggly vs straight the trace is (ignoring turns)
- sprawl - distance (in miles) from most northwest vs most southeast points in the trace
- is_winter - workout in Dec-Feb for northern hemisphere, or July-August for southern
- is_summer - workout in Dec-Feb for southern hemisphere, or July-August for northern

In [1]:
import pandas as pd

df = pd.read_csv("data_for_modeling.csv")

# create winter flag
df['is_winter'] = (
    ((df['is_northern_hemisphere'] == 1) & (df['month'].isin([12, 1, 2]))) |
    ((df['is_northern_hemisphere'] == 0) & (df['month'].isin([6, 7, 8])))
).astype(int)

# create summer flag
df['is_summer'] = (
    ((df['is_northern_hemisphere'] == 1) & (df['month'].isin([6, 7, 8]))) |
    ((df['is_northern_hemisphere'] == 0) & (df['month'].isin([12, 1, 2])))
).astype(int)

### Target variable

sport_type_grouped

### Features to remove

id / user_id / sport_type / start_date / device_name / suffer_score / is_northern_hemisphere

### Feature Selection

* miles <- distance/miles/kilometers
* moving_time <- moving_time/moving_minutes
* elapsed_time <- elapsed_time/elapsed_minutes
* feet_per_mile <- total_elevation_gain/meters_per_km

### Binary Features

commute / manual / has_gear / is_weekend / is_northern_hemisphere

### Categorical -> One-hot encoding

day_part

### GPS features -> combine into binary variable(has_gps)
⁠num_turns / turns_per_mile / wobble / sprawl



In [2]:
df['has_gps'] = df['num_turns'].notna().astype(int)

gps_cols = ['num_turns','turns_per_mile','wobble','sprawl']
df[gps_cols] = df[gps_cols].fillna(0)

df['has_gps'].value_counts()

has_gps
1    342829
0     55084
Name: count, dtype: int64

In [3]:
# Remove Anomalies
df = df[df['is_anomaly']==0].copy()

In [4]:
# Define Target and Features
y = df['sport_type_grouped']

final_feature_list = [
    # Core workout intensity / size
    "speed_mph",
    "miles",
    "moving_time",
    "elapsed_time",
    "moving_time_per",
    "feet_per_mile",

    # Route / GPS-shape features (+ indicator)
    "has_gps",
    "num_turns",
    "turns_per_mile",
    "wobble",
    "sprawl",

    # Time patterns
    "hour",
    "month",
    "dayofweek",
    "is_weekend",

    # Context flags
    "commute",
    "manual",
    "has_gear",
    "is_winter",
    "is_summer",

    # Light engagement signal (optional but usually safe)
    "kudos_count",

    # Categorical (we will one-hot encode later)
    "day_part",
]

X = df[final_feature_list].copy()


In [5]:
# Leakage Sanity Check
leak_cols = {"id", "user_id", "sport_type", "sport_type_grouped", "start_date", "device_name", "suffer_score", "is_anomaly"}
print("Leakage cols in X:", leak_cols.intersection(X.columns))

# Confirm no missing values remain after GPS filling
print(X.isna().sum().sort_values(ascending=False).head(10))


Leakage cols in X: set()
speed_mph      0
miles          0
kudos_count    0
is_summer      0
is_winter      0
has_gear       0
manual         0
commute        0
is_weekend     0
dayofweek      0
dtype: int64


In [6]:
# Encode categorical variables
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify catergorical + numeric columns
categorical_cols = ["day_part"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

In [7]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
sport_type_grouped
Ride       0.419841
Run        0.366082
Workout    0.093403
Walk       0.091773
Hike       0.028901
Name: proportion, dtype: float64

Test distribution:
sport_type_grouped
Ride       0.419838
Run        0.366083
Workout    0.093412
Walk       0.091766
Hike       0.028901
Name: proportion, dtype: float64


In [8]:
# Define Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [9]:
# Scaling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Full pipeline
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("scaling", StandardScaler(with_mean=False)),  # required after one-hot
    ("classifier", LogisticRegression(max_iter=1000))
])

In [10]:
# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {macro_f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix (pretty)
cm = confusion_matrix(y_test, y_pred, labels=model.named_steps["classifier"].classes_)
cm_df = pd.DataFrame(
    cm,
    index=model.named_steps["classifier"].classes_,
    columns=model.named_steps["classifier"].classes_
)
print("\nConfusion Matrix:")
print(cm_df)

Accuracy: 0.8579
Macro F1:  0.7414

Classification Report:
              precision    recall  f1-score   support

        Hike       0.63      0.32      0.42      2300
        Ride       0.93      0.89      0.91     33412
         Run       0.85      0.91      0.88     29134
        Walk       0.69      0.79      0.73      7303
     Workout       0.80      0.73      0.76      7434

    accuracy                           0.86     79583
   macro avg       0.78      0.73      0.74     79583
weighted avg       0.86      0.86      0.86     79583


Confusion Matrix:
         Hike   Ride    Run  Walk  Workout
Hike      726     41    211  1164      158
Ride       16  29861   2780   271      484
Run        98   1394  26552   685      405
Walk      250    101    928  5741      283
Workout    64    649    857   472     5392


# Best_Model_XGBoost

In [19]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [20]:
from xgboost import XGBClassifier

In [21]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

xgb_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", xgb)
])

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [24]:
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGB Macro F1:", f1_score(y_test, y_pred_xgb, average="macro"))
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

XGB Accuracy: 0.9257504743475365
XGB Macro F1: 0.8574242257342075
              precision    recall  f1-score   support

           0       0.71      0.62      0.66      2300
           1       0.97      0.95      0.96     33412
           2       0.93      0.95      0.94     29134
           3       0.81      0.89      0.85      7303
           4       0.90      0.85      0.88      7434

    accuracy                           0.93     79583
   macro avg       0.87      0.85      0.86     79583
weighted avg       0.93      0.93      0.93     79583

[[ 1419    18    86   742    35]
 [   18 31603  1293    81   417]
 [  116   685 27814   377   142]
 [  388    47   274  6499    95]
 [   48   269   493   285  6339]]


In [25]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

param_dist = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [3, 4, 6, 8],
    "classifier__min_child_weight": [1, 5, 10],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "classifier__reg_lambda": [0.5, 1.0, 2.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search_xgb.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(drop='first'),
                                                                               ['day_part']),
                                                                              ('num',
                                                                               'passthrough',
                                                                               ['speed_mph',
                                                                                'miles',
                                                                                'moving_time',
                                                                                'elapsed_time',
                                                                                'moving_time_per',
                                                                                'feet_per_mile',
                                                                                'has_gps',
                                                                                'num_turns',
                                                                                'turns_per_mile',
                                                                                'wo...
                   n_iter=20, n_jobs=-1,
                   param_distributions={'classifier__colsample_bytree': [0.6,
                                                                         0.8,
                                                                         1.0],
                                        'classifier__learning_rate': [0.03,
                                                                      0.05,
                                                                      0.1],
                                        'classifier__max_depth': [3, 4, 6, 8],
                                        'classifier__min_child_weight': [1, 5,
                                                                         10],
                                        'classifier__n_estimators': [300, 500,
                                                                     800],
                                        'classifier__reg_lambda': [0.5, 1.0,
                                                                   2.0],
                                        'classifier__subsample': [0.6, 0.8,
                                                                  1.0]},
                   random_state=42, scoring='f1_macro', verbose=2)

In [26]:
best_xgb = search_xgb.best_estimator_

y_pred_best = best_xgb.predict(X_test)

print("Best XGB params:", search_xgb.best_params_)
print("XGB TEST Accuracy:", accuracy_score(y_test, y_pred_best))
print("XGB TEST Macro F1:", f1_score(y_test, y_pred_best, average="macro"))
print(classification_report(y_test, y_pred_best))

Best XGB params: {'classifier__subsample': 0.6, 'classifier__reg_lambda': 0.5, 'classifier__n_estimators': 800, 'classifier__min_child_weight': 1, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.1, 'classifier__colsample_bytree': 1.0}
XGB TEST Accuracy: 0.933892916828971
XGB TEST Macro F1: 0.8683731521703401
              precision    recall  f1-score   support

           0       0.73      0.62      0.67      2300
           1       0.97      0.95      0.96     33412
           2       0.94      0.96      0.95     29134
           3       0.83      0.90      0.86      7303
           4       0.92      0.88      0.90      7434

    accuracy                           0.93     79583
   macro avg       0.88      0.86      0.87     79583
weighted avg       0.93      0.93      0.93     79583



In [28]:
import joblib
import sklearn, xgboost

bundle = {
    "model": best_xgb,                  
    "label_encoder": le,                
    "feature_cols": X.columns.tolist(), 
    "target_col": "sport_type_grouped",
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__
}

joblib.dump(bundle, "best_xgb_bundle.joblib")
print("✅ Saved: best_xgb_bundle.joblib")

✅ Saved: best_xgb_bundle.joblib


# External Evaluation

In [29]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, log_loss

In [30]:
bundle = joblib.load("best_xgb_bundle.joblib")
model = bundle["model"]
le = bundle["label_encoder"]
feature_cols = bundle["feature_cols"]
target_col = bundle["target_col"] 

In [31]:
df_new = pd.read_csv("data_for_modeling_Mar4.csv") 

In [32]:
df_new = df_new[df_new[target_col].notna()].copy()

train_labels = set(le.classes_)
new_labels = set(df_new[target_col].unique())
unseen_labels = new_labels - train_labels
print("Unseen labels in new data:", unseen_labels)

Unseen labels in new data: {'EBikeRide'}


In [33]:
df_new["sport_type_grouped"] = df_new["sport_type_grouped"].replace({
    "EBikeRide": "Ride"
})

In [36]:
y_new = le.transform(df_new["sport_type_grouped"])

In [37]:
# ----- Feature engineering -----
gps_cols = ["num_turns", "turns_per_mile", "wobble", "sprawl"]

# has_gps = 1 if num_turns is not NaN else 0
df_new["has_gps"] = df_new["num_turns"].notna().astype(int)

# fill NaNs in GPS-shape cols with 0 (as training did)
for c in gps_cols:
    if c in df_new.columns:
        df_new[c] = df_new[c].fillna(0)
    else:
        df_new[c] = 0

In [38]:
# month column preferred 
if "month" in df_new.columns:
    m = df_new["month"]
else:
    df_new["start_date_local"] = pd.to_datetime(df_new["start_date_local"], errors="coerce")
    m = df_new["start_date_local"].dt.month

df_new["is_winter"] = m.isin([12, 1, 2]).astype(int)
df_new["is_summer"] = m.isin([6, 7, 8]).astype(int)

In [39]:
missing_cols = [c for c in feature_cols if c not in df_new.columns]
if missing_cols:
    raise ValueError(f"New data missing columns: {missing_cols}")

X_new = df_new[feature_cols].copy()

In [40]:
y_pred = model.predict(X_new)

In [47]:
print("External Accuracy:", accuracy_score(y_new, y_pred))
print("External Macro F1:", f1_score(y_new, y_pred, average="macro"))
print(classification_report(y_new, y_pred))

External Accuracy: 0.9019438444924406
External Macro F1: 0.8191005763406874
              precision    recall  f1-score   support

           0       0.50      0.47      0.48        49
           1       0.91      0.95      0.93       730
           2       0.91      0.92      0.92       860
           3       0.87      0.87      0.87       342
           4       0.95      0.85      0.90       334

    accuracy                           0.90      2315
   macro avg       0.83      0.81      0.82      2315
weighted avg       0.90      0.90      0.90      2315



In [42]:
y_true_label = le.inverse_transform(y_new)
y_pred_label = le.inverse_transform(y_pred)

result_df = df_new.copy()
result_df["y_true"] = y_true_label
result_df["y_pred"] = y_pred_label
result_df["correct"] = (result_df["y_true"] == result_df["y_pred"])

result_df[["y_true","y_pred","correct"]].head(20)

,y_true,y_pred,correct
0,Walk,Walk,True
1,Workout,Workout,True
2,Run,Run,True
3,Ride,Ride,True
4,Run,Run,True
5,Run,Run,True
6,Workout,Workout,True
7,Run,Run,True
8,Run,Run,True
9,Run,Run,True


In [46]:
labels = le.classes_ 
cm = confusion_matrix(y_new, y_pred)

cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in labels],
                        columns=[f"Pred_{c}" for c in labels])

cm_df

,Pred_Hike,Pred_Ride,Pred_Run,Pred_Walk,Pred_Workout
True_Hike,23,1,0,24,1
True_Ride,0,692,29,1,8
True_Run,5,48,792,11,4
True_Walk,14,4,25,296,3
True_Workout,4,14,23,8,285


In [44]:
errors = result_df[~result_df["correct"]]
errors.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False).head(15)

y_true   y_pred 
Run      Ride       48
Ride     Run        29
Walk     Run        25
Hike     Walk       24
Workout  Run        23
         Ride       14
Walk     Hike       14
Run      Walk       11
Workout  Walk        8
Ride     Workout     8
Run      Hike        5
Workout  Hike        4
Run      Workout     4
Walk     Ride        4
         Workout     3
dtype: int64